# Chapter 4 — Controlled Runs, Queued Input, and Cancellation

This chapter turns the structured Tool loop into a controllable long-running AgentSession while preserving deterministic conversation state.

## Goal and Previous Limitation

Chapter 3 can complete a model–Tool loop, but callers can start overlapping work against the same mutable history, cannot redirect an active Run, and lose the distinction between queued work and cancellation settlement. We begin from the immutable Chapter 3 Checkpoint and make that missing control layer observable.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
CHAPTER_3 = ROOT / 'course' / 'checkpoints' / 'ch03'
sys.path.insert(0, str(CHAPTER_3 / 'src'))
import agent_harness as chapter3

assert hasattr(chapter3, 'AgentRuntime')
assert not hasattr(chapter3, 'AgentSession')
for module_name in tuple(sys.modules):
    if module_name == 'agent_harness' or module_name.startswith('agent_harness.'):
        del sys.modules[module_name]
sys.path.pop(0)


## Conceptual Model

`AgentSession` is the new deep module and lifecycle seam. Its small interface—`start`, `run`/`prompt`, `steer`, `follow_up`, `cancel`, and `busy`—hides queue ownership, one-active-Run enforcement, Event forwarding, and pending-input restoration. `AgentRuntime` remains independently usable and gains only a `TurnInput` seam plus a typed optional `RunGuard`.

A Steering Message belongs to the active Run and is accepted only at a settled model-turn boundary. A Follow-up Message belongs to the Session queue and becomes a new Run only after `agent_end`. Cancellation is a settlement protocol: provider and Tool tasks are interrupted, every Tool Call receives an ordered result, and only then does the Session become idle.

## Minimal Execution

The next Export Cells replace the evolved Runtime and Tool modules, add the Session module, and publish the complete Chapter 4 interface. Each cell owns one complete source file.

In [ ]:
TOOLS_SOURCE = '"""Explicit Tools and their replaceable execution seam."""\n\nfrom __future__ import annotations\n\nimport asyncio\nfrom collections.abc import Awaitable, Callable, Mapping, Sequence\nfrom dataclasses import dataclass, field\nfrom enum import Enum\nfrom types import MappingProxyType\nfrom typing import Protocol\nfrom pathlib import Path\n\nfrom jsonschema import Draft202012Validator  # type: ignore[import-untyped]\n\nfrom .model import ModelToolResultMessage, ToolCallContent, ToolDefinition\n\n\nToolHandler = Callable[[dict[str, object]], Awaitable["ToolResult"]]\n\n\nclass ToolErrorCode(str, Enum):\n    INVALID_JSON = "invalid_json"\n    INVALID_ARGUMENTS = "invalid_arguments"\n    UNKNOWN_TOOL = "unknown_tool"\n    EXECUTION_FAILED = "execution_failed"\n    CANCELLED = "cancelled"\n\n\nclass TruncationDirection(str, Enum):\n    HEAD = "head"\n    TAIL = "tail"\n\n\nclass CompleteOutputKind(str, Enum):\n    ARTIFACT = "artifact"\n    EXTERNAL = "external"\n    UNAVAILABLE = "unavailable"\n\n\n@dataclass(frozen=True, slots=True)\nclass CompleteOutputReference:\n    kind: CompleteOutputKind\n    reference: str | None = None\n    reason: str | None = None\n\n    @classmethod\n    def artifact(cls, reference: str) -> "CompleteOutputReference":\n        return cls(CompleteOutputKind.ARTIFACT, reference=reference)\n\n    @classmethod\n    def unavailable(cls) -> "CompleteOutputReference":\n        return cls(\n            CompleteOutputKind.UNAVAILABLE,\n            reason="complete output was not retained",\n        )\n\n    def render(self) -> str:\n        if self.reference is not None:\n            return f"{self.kind.value} {self.reference}"\n        return f"{self.kind.value} ({self.reason})"\n\n\n@dataclass(frozen=True, slots=True)\nclass TruncationNotice:\n    original_bytes: int\n    original_lines: int\n    retained_start_byte: int\n    retained_end_byte: int\n    retained_start_line: int\n    retained_end_line: int\n    direction: TruncationDirection\n\n\n@dataclass(frozen=True, slots=True)\nclass ToolOutputBudget:\n    max_bytes: int = 50 * 1024\n    max_lines: int = 2000\n\n    def __post_init__(self) -> None:\n        if self.max_bytes <= 0 or self.max_lines <= 0:\n            raise ValueError("Tool output limits must be positive")\n\n\n@dataclass(frozen=True, slots=True)\nclass ToolResult:\n    content: str\n    metadata: Mapping[str, object] = field(default_factory=dict)\n    terminate: bool = False\n    is_error: bool = False\n    error_code: ToolErrorCode | None = None\n    truncation: TruncationNotice | None = None\n    complete_output: CompleteOutputReference | None = None\n\n    def __post_init__(self) -> None:\n        object.__setattr__(self, "metadata", MappingProxyType(dict(self.metadata)))\n        if self.is_error != (self.error_code is not None):\n            raise ValueError("error ToolResult values require exactly one error_code")\n        if self.is_error and self.terminate:\n            raise ValueError("error ToolResult values cannot terminate a Tool Batch")\n\n    @classmethod\n    def error(cls, code: ToolErrorCode, tool_name: str) -> "ToolResult":\n        guidance = {\n            ToolErrorCode.INVALID_JSON: "provide one valid JSON object",\n            ToolErrorCode.INVALID_ARGUMENTS: "match the Tool\'s declared input schema",\n            ToolErrorCode.UNKNOWN_TOOL: "choose one of the advertised Tools",\n            ToolErrorCode.EXECUTION_FAILED: "revise the call or choose another Tool",\n            ToolErrorCode.CANCELLED: "retry after the cancelled Run is settled",\n        }[code]\n        return cls(\n            content=f"Tool error [{code.value}] for \'{tool_name}\': {guidance}.",\n            is_error=True,\n            error_code=code,\n        )\n\n\n@dataclass(frozen=True, slots=True)\nclass Tool:\n    name: str\n    description: str\n    input_schema: Mapping[str, object]\n    execute: ToolHandler\n    sequential: bool = False\n    output_direction: TruncationDirection = TruncationDirection.HEAD\n\n    def __post_init__(self) -> None:\n        if not self.name.strip():\n            raise ValueError("Tool name cannot be empty")\n        if not self.description.strip():\n            raise ValueError("Tool description cannot be empty")\n        if not callable(self.execute):\n            raise TypeError("Tool execute must be an async callable")\n        schema = dict(self.input_schema)\n        Draft202012Validator.check_schema(schema)\n        if schema.get("type") != "object":\n            raise ValueError("Tool input_schema must describe a JSON object")\n        object.__setattr__(self, "input_schema", MappingProxyType(schema))\n\n    def definition(self) -> ToolDefinition:\n        return ToolDefinition(self.name, self.description, self.input_schema)\n\n\n@dataclass(frozen=True, slots=True)\nclass PreparedToolCall:\n    call: ToolCallContent\n    tool: Tool\n    arguments: dict[str, object]\n\n\n@dataclass(frozen=True, slots=True)\nclass ToolResultMessage:\n    tool_call_id: str\n    tool_name: str\n    result: ToolResult\n\n    def to_model(self) -> ModelToolResultMessage:\n        return ModelToolResultMessage(\n            self.tool_call_id,\n            self.tool_name,\n            self.result.content,\n            self.result.is_error,\n        )\n\n\nclass ToolExecutor(Protocol):\n    async def execute(self, call: PreparedToolCall) -> ToolResult: ...\n\n\nclass LocalToolExecutor:\n    async def execute(self, call: PreparedToolCall) -> ToolResult:\n        return await call.tool.execute(call.arguments)\n\n\n@dataclass(frozen=True, slots=True)\nclass ProcessResult:\n    returncode: int\n    stdout: str\n    stderr: str\n\n\nclass AsyncioProcessOperations:\n    """Run an argv directly and terminate the child when its task is cancelled.\n\n    This is host-process execution infrastructure, not a sandbox.\n    """\n\n    async def run(\n        self,\n        command: Sequence[str],\n        *,\n        cwd: str | Path | None = None,\n        timeout_seconds: float | None = None,\n    ) -> ProcessResult:\n        argv = tuple(command)\n        if not argv or any(not isinstance(part, str) or not part for part in argv):\n            raise ValueError("command must contain non-empty argv strings")\n        if timeout_seconds is not None and timeout_seconds <= 0:\n            raise ValueError("timeout_seconds must be positive when supplied")\n        process = await asyncio.create_subprocess_exec(\n            *argv,\n            cwd=cwd,\n            stdout=asyncio.subprocess.PIPE,\n            stderr=asyncio.subprocess.PIPE,\n        )\n        try:\n            communication = process.communicate()\n            if timeout_seconds is None:\n                stdout, stderr = await communication\n            else:\n                stdout, stderr = await asyncio.wait_for(\n                    communication, timeout_seconds\n                )\n        except (asyncio.CancelledError, TimeoutError):\n            if process.returncode is None:\n                process.terminate()\n                try:\n                    await asyncio.wait_for(process.wait(), timeout=1)\n                except TimeoutError:\n                    process.kill()\n                    await process.wait()\n            raise\n        assert process.returncode is not None\n        return ProcessResult(\n            process.returncode,\n            stdout.decode("utf-8", errors="replace"),\n            stderr.decode("utf-8", errors="replace"),\n        )\n\n\ndef bound_tool_result(\n    result: ToolResult,\n    budget: ToolOutputBudget,\n    direction: TruncationDirection,\n) -> ToolResult:\n    """Bound model-facing text and attach explicit recovery provenance."""\n\n    content = result.content\n    encoded = content.encode("utf-8")\n    lines = content.splitlines(keepends=True)\n    original_lines = len(content.splitlines())\n    if len(encoded) <= budget.max_bytes and original_lines <= budget.max_lines:\n        return result\n\n    if direction is TruncationDirection.HEAD:\n        line_limited = "".join(lines[: budget.max_lines])\n        retained_bytes = line_limited.encode("utf-8")[: budget.max_bytes]\n        retained = retained_bytes.decode("utf-8", errors="ignore")\n        retained_start_byte = 0\n        retained_end_byte = len(retained.encode("utf-8"))\n        retained_start_line = 1 if retained else 0\n        retained_end_line = len(retained.splitlines())\n    else:\n        line_limited = "".join(lines[-budget.max_lines :])\n        retained_bytes = line_limited.encode("utf-8")[-budget.max_bytes :]\n        retained = retained_bytes.decode("utf-8", errors="ignore")\n        retained_end_byte = len(encoded)\n        retained_start_byte = retained_end_byte - len(retained.encode("utf-8"))\n        retained_end_line = original_lines\n        retained_line_count = len(retained.splitlines())\n        retained_start_line = max(1, original_lines - retained_line_count + 1)\n\n    reference = result.complete_output or CompleteOutputReference.unavailable()\n    notice = TruncationNotice(\n        original_bytes=len(encoded),\n        original_lines=original_lines,\n        retained_start_byte=retained_start_byte,\n        retained_end_byte=retained_end_byte,\n        retained_start_line=retained_start_line,\n        retained_end_line=retained_end_line,\n        direction=direction,\n    )\n    model_notice = (\n        "[tool output truncated: "\n        f"retained {direction.value} bytes {retained_start_byte}-{retained_end_byte} "\n        f"of {len(encoded)}, lines {retained_start_line}-{retained_end_line} "\n        f"of {original_lines}; complete output: {reference.render()}]"\n    )\n    return ToolResult(\n        content=f"{retained}\\n\\n{model_notice}",\n        metadata=result.metadata,\n        terminate=result.terminate,\n        is_error=result.is_error,\n        error_code=result.error_code,\n        truncation=notice,\n        complete_output=reference,\n    )\n'


In [ ]:
RUNTIME_SOURCE = '"""Async Agent Runtime with structured Tool batches."""\n\nfrom __future__ import annotations\n\nimport asyncio\nfrom collections.abc import AsyncIterator, Awaitable, Callable, Sequence\nfrom dataclasses import dataclass\nfrom enum import Enum\nimport json\nfrom typing import Protocol, TypeAlias, cast\n\nfrom jsonschema import (  # type: ignore[import-untyped]\n    Draft202012Validator,\n    ValidationError,\n)\n\nfrom .model import (\n    AgentMessage,\n    ContentBlock,\n    ModelAdapter,\n    ModelAdapterError,\n    ModelEnd,\n    ModelError,\n    ModelErrorCode,\n    ModelEvent,\n    ModelRequest,\n    ModelSpec,\n    Role,\n    StopReason,\n    TextContent,\n    TextDelta,\n    ToolCallContent,\n    ToolCallDelta,\n    Usage,\n    UsageUpdate,\n    to_model_messages,\n)\nfrom .tools import (\n    LocalToolExecutor,\n    PreparedToolCall,\n    Tool,\n    ToolErrorCode,\n    ToolExecutor,\n    ToolOutputBudget,\n    ToolResult,\n    ToolResultMessage,\n    bound_tool_result,\n)\n\n\nclass EventType(str, Enum):\n    AGENT_START = "agent_start"\n    MODEL_ATTEMPT_START = "model_attempt_start"\n    MODEL_EVENT = "model_event"\n    MODEL_ATTEMPT_FAILED = "model_attempt_failed"\n    RETRY_SCHEDULED = "retry_scheduled"\n    TOOL_BATCH_START = "tool_batch_start"\n    TOOL_CALL_START = "tool_call_start"\n    TOOL_CALL_END = "tool_call_end"\n    TOOL_BATCH_END = "tool_batch_end"\n    RUN_CANCELLED = "run_cancelled"\n    MESSAGE_END = "message_end"\n    AGENT_END = "agent_end"\n\n\nclass TerminalStatus(str, Enum):\n    COMPLETED = "completed"\n    MODEL_ERROR = "model_error"\n    CANCELLED = "cancelled"\n    MAX_TURNS = "max_turns"\n    MAX_TOOL_CALLS = "max_tool_calls"\n    TIMEOUT = "timeout"\n    MAX_TOTAL_TOKENS = "max_total_tokens"\n\n\n@dataclass(frozen=True, slots=True)\nclass RunGuard:\n    max_turns: int | None = None\n    max_tool_calls: int | None = None\n    timeout_seconds: float | None = None\n    max_total_tokens: int | None = None\n\n    def __post_init__(self) -> None:\n        integer_limits = {\n            "max_turns": self.max_turns,\n            "max_tool_calls": self.max_tool_calls,\n            "max_total_tokens": self.max_total_tokens,\n        }\n        for name, value in integer_limits.items():\n            if value is not None and (\n                isinstance(value, bool) or not isinstance(value, int) or value <= 0\n            ):\n                raise ValueError(f"{name} must be a positive integer when supplied")\n        if self.timeout_seconds is not None and self.timeout_seconds <= 0:\n            raise ValueError("timeout_seconds must be positive when supplied")\n\n    def reached(\n        self,\n        *,\n        turns: int,\n        tool_calls: int,\n        usage: Usage | None,\n        elapsed_seconds: float,\n    ) -> TerminalStatus | None:\n        if (\n            self.timeout_seconds is not None\n            and elapsed_seconds >= self.timeout_seconds\n        ):\n            return TerminalStatus.TIMEOUT\n        if self.max_turns is not None and turns >= self.max_turns:\n            return TerminalStatus.MAX_TURNS\n        if (\n            self.max_tool_calls is not None\n            and tool_calls >= self.max_tool_calls\n        ):\n            return TerminalStatus.MAX_TOOL_CALLS\n        if (\n            self.max_total_tokens is not None\n            and usage is not None\n            and usage.total_tokens >= self.max_total_tokens\n        ):\n            return TerminalStatus.MAX_TOTAL_TOKENS\n        return None\n\n\n@dataclass(frozen=True, slots=True)\nclass RuntimeEvent:\n    sequence: int\n    type: EventType\n    attempt: int | None = None\n    model_event: ModelEvent | None = None\n    error: ModelError | None = None\n    retry_delay_seconds: float | None = None\n    partial_text: str = ""\n    partial_usage: Usage | None = None\n    tool_call_id: str | None = None\n    tool_name: str | None = None\n    tool_result: ToolResult | None = None\n\n\n@dataclass(frozen=True, slots=True)\nclass AssistantOutcome:\n    message: AgentMessage\n    stop_reason: StopReason\n    usage: Usage | None = None\n    error: ModelError | None = None\n    attempts: int = 1\n    tool_results: tuple[ToolResult, ...] = ()\n    status: TerminalStatus = TerminalStatus.COMPLETED\n\n\n@dataclass(frozen=True, slots=True)\nclass RetryPolicy:\n    delays: tuple[float, ...] = (2.0, 4.0, 8.0)\n    max_retry_after_seconds: float = 60.0\n\n    def __post_init__(self) -> None:\n        if any(delay < 0 for delay in self.delays):\n            raise ValueError("retry delays cannot be negative")\n        if self.max_retry_after_seconds < 0:\n            raise ValueError("max_retry_after_seconds cannot be negative")\n\n    def delay_for(\n        self,\n        error: ModelError,\n        failed_attempt: int,\n        *,\n        retry_after_seconds: float | None = None,\n    ) -> float | None:\n        retryable_codes = {\n            ModelErrorCode.RATE_LIMIT,\n            ModelErrorCode.TIMEOUT,\n            ModelErrorCode.CONNECTION,\n            ModelErrorCode.SERVER,\n        }\n        retryable_status = error.status_code in {408, 429} or (\n            error.status_code is not None and error.status_code >= 500\n        )\n        if (\n            error.code not in retryable_codes and not retryable_status\n        ) or failed_attempt > len(self.delays):\n            return None\n        if retry_after_seconds is not None:\n            if not 0 <= retry_after_seconds <= self.max_retry_after_seconds:\n                return None\n            return retry_after_seconds\n        return self.delays[failed_attempt - 1]\n\n\nSleeper: TypeAlias = Callable[[float], Awaitable[None]]\nConversationMessage: TypeAlias = AgentMessage | ToolResultMessage\n_EVENTS_DONE = object()\n\n\nclass TurnInput(Protocol):\n    """Runtime-facing view of Steering Messages waiting at a turn boundary."""\n\n    def pending(self) -> bool: ...\n\n    def take(self) -> Sequence[AgentMessage]: ...\n\n\n@dataclass(slots=True)\nclass _CancellationState:\n    status: TerminalStatus = TerminalStatus.CANCELLED\n    requested: bool = False\n\n    def request(self, status: TerminalStatus) -> bool:\n        if self.requested:\n            return False\n        self.status = status\n        self.requested = True\n        return True\n\n\ndef _add_usage(left: Usage | None, right: Usage | None) -> Usage | None:\n    if left is None:\n        return right\n    if right is None:\n        return left\n    return Usage(\n        left.input_tokens + right.input_tokens,\n        left.output_tokens + right.output_tokens,\n        left.total_tokens + right.total_tokens,\n        left.estimated or right.estimated,\n    )\n\n\nclass AgentRunHandle:\n    """One accepted run\'s observations, cancellation, and eventual outcome."""\n\n    def __init__(\n        self,\n        task: asyncio.Task[AssistantOutcome],\n        events: asyncio.Queue[RuntimeEvent | object],\n        cancellation: _CancellationState,\n    ) -> None:\n        self._task = task\n        self._events = events\n        self._cancellation = cancellation\n\n    async def events(self) -> AsyncIterator[RuntimeEvent]:\n        while True:\n            event = await self._events.get()\n            if event is _EVENTS_DONE:\n                break\n            yield cast(RuntimeEvent, event)\n\n    async def result(self) -> AssistantOutcome:\n        return await self._task\n\n    def cancel(self) -> None:\n        self._cancel_with(TerminalStatus.CANCELLED)\n\n    def _cancel_with(self, status: TerminalStatus) -> None:\n        if not self._task.done() and self._cancellation.request(status):\n            self._task.get_loop().call_soon(self._task.cancel)\n\n\nclass AgentRuntime:\n    """Advance typed conversation state through model and Tool turns."""\n\n    def __init__(\n        self,\n        adapter: ModelAdapter,\n        model: ModelSpec,\n        *,\n        tools: Sequence[Tool] = (),\n        tool_executor: ToolExecutor | None = None,\n        tool_output_budget: ToolOutputBudget | None = None,\n        retry_policy: RetryPolicy | None = None,\n        sleeper: Sleeper = asyncio.sleep,\n        run_guard: object | None = None,\n    ) -> None:\n        if not isinstance(model, ModelSpec):\n            raise TypeError("model must be a ModelSpec")\n        if not callable(getattr(adapter, "stream", None)):\n            raise TypeError("adapter must implement ModelAdapter.stream")\n        if retry_policy is not None and not isinstance(retry_policy, RetryPolicy):\n            raise TypeError("retry_policy must be a RetryPolicy")\n        if not callable(sleeper):\n            raise TypeError("sleeper must be an async callable")\n        if tool_executor is not None and not callable(\n            getattr(tool_executor, "execute", None)\n        ):\n            raise TypeError("tool_executor must implement ToolExecutor.execute")\n        if tool_output_budget is not None and not isinstance(\n            tool_output_budget, ToolOutputBudget\n        ):\n            raise TypeError("tool_output_budget must be a ToolOutputBudget")\n        registered: dict[str, Tool] = {}\n        for tool in tools:\n            if not isinstance(tool, Tool):\n                raise TypeError("tools must contain Tool values")\n            if tool.name in registered:\n                raise ValueError(f"duplicate Tool name: {tool.name!r}")\n            registered[tool.name] = tool\n        if registered and not model.supports_tools:\n            raise ValueError("configured ModelSpec does not support Tools")\n        self._adapter = adapter\n        self._model = model\n        self._tools = registered\n        self._tool_executor = tool_executor or LocalToolExecutor()\n        self._tool_output_budget = tool_output_budget or ToolOutputBudget()\n        self._retry_policy = retry_policy or RetryPolicy()\n        self._sleeper = sleeper\n        self._run_guard = run_guard\n        self._history: list[ConversationMessage] = []\n\n    @property\n    def history(self) -> tuple[ConversationMessage, ...]:\n        return tuple(self._history)\n\n    @property\n    def run_guard(self) -> object | None:\n        return self._run_guard\n\n    def start(\n        self,\n        messages: Sequence[AgentMessage],\n        *,\n        turn_input: TurnInput | None = None,\n    ) -> AgentRunHandle:\n        accepted = tuple(messages)\n        to_model_messages((*self._history, *accepted))\n        self._history.extend(accepted)\n        events: asyncio.Queue[RuntimeEvent | object] = asyncio.Queue()\n        cancellation = _CancellationState()\n        task = asyncio.get_running_loop().create_task(\n            self._execute(events, cancellation, turn_input)\n        )\n        handle = AgentRunHandle(task, events, cancellation)\n        if (\n            isinstance(self._run_guard, RunGuard)\n            and self._run_guard.timeout_seconds is not None\n        ):\n            timer = asyncio.get_running_loop().call_later(\n                self._run_guard.timeout_seconds,\n                handle._cancel_with,\n                TerminalStatus.TIMEOUT,\n            )\n            task.add_done_callback(lambda completed: timer.cancel())\n        return handle\n\n    async def run(self, messages: Sequence[AgentMessage]) -> AssistantOutcome:\n        return await self.start(messages).result()\n\n    async def _execute(\n        self,\n        events: asyncio.Queue[RuntimeEvent | object],\n        cancellation: _CancellationState,\n        turn_input: TurnInput | None = None,\n    ) -> AssistantOutcome:\n        sequence = 0\n        total_attempts = 0\n        text_parts: list[str] = []\n        current_usage: Usage | None = None\n        run_usage: Usage | None = None\n        run_tool_results: list[ToolResult] = []\n        turns = 0\n        tool_calls = 0\n        started_at = asyncio.get_running_loop().time()\n\n        def guard_status() -> TerminalStatus | None:\n            if not isinstance(self._run_guard, RunGuard):\n                return None\n            return self._run_guard.reached(\n                turns=turns,\n                tool_calls=tool_calls,\n                usage=run_usage,\n                elapsed_seconds=asyncio.get_running_loop().time() - started_at,\n            )\n\n        async def emit(\n            type_: EventType,\n            *,\n            attempt: int | None = None,\n            model_event: ModelEvent | None = None,\n            error: ModelError | None = None,\n            retry_delay_seconds: float | None = None,\n            partial_text: str = "",\n            partial_usage: Usage | None = None,\n            tool_call_id: str | None = None,\n            tool_name: str | None = None,\n            tool_result: ToolResult | None = None,\n        ) -> None:\n            nonlocal sequence\n            sequence += 1\n            await events.put(\n                RuntimeEvent(\n                    sequence=sequence,\n                    type=type_,\n                    attempt=attempt,\n                    model_event=model_event,\n                    error=error,\n                    retry_delay_seconds=retry_delay_seconds,\n                    partial_text=partial_text,\n                    partial_usage=partial_usage,\n                    tool_call_id=tool_call_id,\n                    tool_name=tool_name,\n                    tool_result=tool_result,\n                )\n            )\n\n        async def finish(outcome: AssistantOutcome) -> AssistantOutcome:\n            self._history.append(outcome.message)\n            await emit(EventType.MESSAGE_END, attempt=max(total_attempts, 1))\n            await emit(EventType.AGENT_END, attempt=max(total_attempts, 1))\n            return outcome\n\n        try:\n            await emit(EventType.AGENT_START)\n            while True:\n                turn_attempt = 0\n                while True:\n                    turn_attempt += 1\n                    total_attempts += 1\n                    attempt = total_attempts\n                    request = ModelRequest(\n                        to_model_messages(self._history),\n                        self._model,\n                        tuple(tool.definition() for tool in self._tools.values()),\n                    )\n                    await emit(EventType.MODEL_ATTEMPT_START, attempt=attempt)\n                    text_parts = []\n                    tool_drafts: dict[int, dict[str, str]] = {}\n                    current_usage = None\n                    end: ModelEnd | None = None\n                    schema_error: ModelError | None = None\n                    try:\n                        async for event in self._adapter.stream(request):\n                            await emit(\n                                EventType.MODEL_EVENT,\n                                attempt=attempt,\n                                model_event=event,\n                            )\n                            if end is not None:\n                                schema_error = ModelError(\n                                    ModelErrorCode.SCHEMA,\n                                    "model stream emitted data after ModelEnd",\n                                    False,\n                                )\n                                break\n                            if isinstance(event, TextDelta):\n                                text_parts.append(event.text)\n                            elif isinstance(event, ToolCallDelta):\n                                if event.index < 0:\n                                    schema_error = ModelError(\n                                        ModelErrorCode.SCHEMA,\n                                        "model stream emitted an invalid Tool Call index",\n                                        False,\n                                    )\n                                    break\n                                draft = tool_drafts.setdefault(\n                                    event.index,\n                                    {"id": "", "name": "", "arguments": ""},\n                                )\n                                draft["id"] += event.id\n                                draft["name"] += event.name\n                                draft["arguments"] += event.arguments_delta\n                            elif isinstance(event, UsageUpdate):\n                                current_usage = event.usage\n                            elif isinstance(event, ModelEnd):\n                                end = event\n                            else:\n                                schema_error = ModelError(\n                                    ModelErrorCode.SCHEMA,\n                                    "model stream emitted an unsupported event",\n                                    False,\n                                )\n                                break\n                    except ModelAdapterError as failure:\n                        partial_text = "".join(text_parts)\n                        await emit(\n                            EventType.MODEL_ATTEMPT_FAILED,\n                            attempt=attempt,\n                            error=failure.error,\n                            partial_text=partial_text,\n                            partial_usage=current_usage,\n                        )\n                        delay = self._retry_policy.delay_for(\n                            failure.error,\n                            turn_attempt,\n                            retry_after_seconds=failure.error.retry_after_seconds,\n                        )\n                        if delay is not None:\n                            await emit(\n                                EventType.RETRY_SCHEDULED,\n                                attempt=attempt,\n                                error=failure.error,\n                                retry_delay_seconds=delay,\n                                partial_text=partial_text,\n                                partial_usage=current_usage,\n                            )\n                            text_parts = []\n                            current_usage = None\n                            await self._sleeper(delay)\n                            continue\n                        terminal_usage = _add_usage(run_usage, current_usage)\n                        return await finish(\n                            AssistantOutcome(\n                                AgentMessage.text(Role.ASSISTANT, partial_text),\n                                StopReason.ERROR,\n                                terminal_usage,\n                                failure.error,\n                                total_attempts,\n                                tuple(run_tool_results),\n                                TerminalStatus.MODEL_ERROR,\n                            )\n                        )\n\n                    error = schema_error\n                    if error is None and end is None:\n                        error = ModelError(\n                            ModelErrorCode.SCHEMA,\n                            "model stream violated the provider-neutral event contract",\n                            False,\n                        )\n                    blocks: list[ContentBlock] = []\n                    if error is None:\n                        try:\n                            if text_parts:\n                                blocks.append(TextContent("".join(text_parts)))\n                            for index in sorted(tool_drafts):\n                                blocks.append(ToolCallContent(**tool_drafts[index]))\n                        except (TypeError, ValueError):\n                            error = ModelError(\n                                ModelErrorCode.SCHEMA,\n                                "model stream emitted an incomplete Tool Call",\n                                False,\n                            )\n                    if error is not None:\n                        partial_text = "".join(text_parts)\n                        await emit(\n                            EventType.MODEL_ATTEMPT_FAILED,\n                            attempt=attempt,\n                            error=error,\n                            partial_text=partial_text,\n                            partial_usage=current_usage,\n                        )\n                        terminal_usage = _add_usage(run_usage, current_usage)\n                        return await finish(\n                            AssistantOutcome(\n                                AgentMessage.text(Role.ASSISTANT, partial_text),\n                                StopReason.ERROR,\n                                terminal_usage,\n                                error,\n                                total_attempts,\n                                tuple(run_tool_results),\n                                TerminalStatus.MODEL_ERROR,\n                            )\n                        )\n                    assert end is not None\n                    break\n\n                run_usage = _add_usage(run_usage, current_usage)\n                turns += 1\n                assistant = AgentMessage(Role.ASSISTANT, tuple(blocks))\n                self._history.append(assistant)\n                await emit(EventType.MESSAGE_END, attempt=total_attempts)\n                calls = tuple(\n                    block\n                    for block in assistant.content\n                    if isinstance(block, ToolCallContent)\n                )\n                if not calls:\n                    if turn_input is not None and turn_input.pending():\n                        reached = guard_status()\n                        if reached is not None:\n                            await emit(EventType.AGENT_END, attempt=total_attempts)\n                            return AssistantOutcome(\n                                assistant,\n                                StopReason.ABORTED,\n                                run_usage,\n                                attempts=total_attempts,\n                                tool_results=tuple(run_tool_results),\n                                status=reached,\n                            )\n                        steering = tuple(turn_input.take())\n                        to_model_messages(steering)\n                        self._history.extend(steering)\n                        continue\n                    await emit(EventType.AGENT_END, attempt=total_attempts)\n                    return AssistantOutcome(\n                        assistant,\n                        end.stop_reason,\n                        run_usage,\n                        attempts=total_attempts,\n                        tool_results=tuple(run_tool_results),\n                    )\n\n                await emit(EventType.TOOL_BATCH_START, attempt=total_attempts)\n                prepared: dict[int, PreparedToolCall] = {}\n                results: dict[int, ToolResult] = {}\n                for index, call in enumerate(calls):\n                    tool = self._tools.get(call.name)\n                    if tool is None:\n                        results[index] = ToolResult.error(\n                            ToolErrorCode.UNKNOWN_TOOL, call.name\n                        )\n                        continue\n                    try:\n                        parsed = json.loads(call.arguments)\n                    except (json.JSONDecodeError, TypeError):\n                        results[index] = ToolResult.error(\n                            ToolErrorCode.INVALID_JSON, call.name\n                        )\n                        continue\n                    try:\n                        Draft202012Validator(tool.input_schema).validate(parsed)\n                    except ValidationError:\n                        results[index] = ToolResult.error(\n                            ToolErrorCode.INVALID_ARGUMENTS, call.name\n                        )\n                        continue\n                    if not isinstance(parsed, dict):\n                        results[index] = ToolResult.error(\n                            ToolErrorCode.INVALID_ARGUMENTS, call.name\n                        )\n                        continue\n                    prepared[index] = PreparedToolCall(call, tool, parsed)\n\n                async def execute_one(index: int, call: PreparedToolCall) -> None:\n                    await emit(\n                        EventType.TOOL_CALL_START,\n                        attempt=total_attempts,\n                        tool_call_id=call.call.id,\n                        tool_name=call.call.name,\n                    )\n                    try:\n                        result = await self._tool_executor.execute(call)\n                        if not isinstance(result, ToolResult):\n                            raise TypeError("ToolExecutor returned an invalid result")\n                    except Exception:\n                        result = ToolResult.error(\n                            ToolErrorCode.EXECUTION_FAILED, call.call.name\n                        )\n                    result = bound_tool_result(\n                        result,\n                        self._tool_output_budget,\n                        call.tool.output_direction,\n                    )\n                    results[index] = result\n                    await emit(\n                        EventType.TOOL_CALL_END,\n                        attempt=total_attempts,\n                        tool_call_id=call.call.id,\n                        tool_name=call.call.name,\n                        tool_result=result,\n                    )\n\n                try:\n                    if any(call.tool.sequential for call in prepared.values()):\n                        for index, prepared_call in prepared.items():\n                            await execute_one(index, prepared_call)\n                    else:\n                        tasks = {\n                            index: asyncio.create_task(\n                                execute_one(index, prepared_call)\n                            )\n                            for index, prepared_call in prepared.items()\n                        }\n                        try:\n                            await asyncio.gather(*tasks.values())\n                        except asyncio.CancelledError:\n                            for task in tasks.values():\n                                if not task.done():\n                                    task.cancel()\n                            await asyncio.gather(\n                                *tasks.values(), return_exceptions=True\n                            )\n                            raise\n                except asyncio.CancelledError:\n                    for index, prepared_call in prepared.items():\n                        if index not in results:\n                            cancelled_result = ToolResult.error(\n                                ToolErrorCode.CANCELLED,\n                                prepared_call.call.name,\n                            )\n                            results[index] = cancelled_result\n                            await emit(\n                                EventType.TOOL_CALL_END,\n                                attempt=total_attempts,\n                                tool_call_id=prepared_call.call.id,\n                                tool_name=prepared_call.call.name,\n                                tool_result=cancelled_result,\n                            )\n                    for index, call in enumerate(calls):\n                        result = results[index]\n                        run_tool_results.append(result)\n                        self._history.append(\n                            ToolResultMessage(call.id, call.name, result)\n                        )\n                    await emit(EventType.TOOL_BATCH_END, attempt=total_attempts)\n                    raise\n\n                batch_results: list[ToolResult] = []\n                for index, call in enumerate(calls):\n                    result = results[index]\n                    batch_results.append(result)\n                    run_tool_results.append(result)\n                    self._history.append(\n                        ToolResultMessage(call.id, call.name, result)\n                    )\n                tool_calls += len(calls)\n                await emit(EventType.TOOL_BATCH_END, attempt=total_attempts)\n                if batch_results and all(result.terminate for result in batch_results):\n                    await emit(EventType.AGENT_END, attempt=total_attempts)\n                    return AssistantOutcome(\n                        assistant,\n                        end.stop_reason,\n                        run_usage,\n                        attempts=total_attempts,\n                        tool_results=tuple(run_tool_results),\n                    )\n                reached = guard_status()\n                if reached is not None:\n                    await emit(EventType.AGENT_END, attempt=total_attempts)\n                    return AssistantOutcome(\n                        assistant,\n                        StopReason.ABORTED,\n                        run_usage,\n                        attempts=total_attempts,\n                        tool_results=tuple(run_tool_results),\n                        status=reached,\n                    )\n                if turn_input is not None and turn_input.pending():\n                    steering = tuple(turn_input.take())\n                    to_model_messages(steering)\n                    self._history.extend(steering)\n        except asyncio.CancelledError:\n            message = AgentMessage.text(Role.ASSISTANT, "".join(text_parts))\n            outcome = AssistantOutcome(\n                message,\n                StopReason.ABORTED,\n                _add_usage(run_usage, current_usage),\n                attempts=max(total_attempts, 1),\n                tool_results=tuple(run_tool_results),\n                status=cancellation.status,\n            )\n            self._history.append(message)\n            await emit(\n                EventType.RUN_CANCELLED,\n                attempt=max(total_attempts, 1),\n                partial_text="".join(text_parts),\n                partial_usage=current_usage,\n            )\n            await emit(EventType.MESSAGE_END, attempt=max(total_attempts, 1))\n            await emit(EventType.AGENT_END, attempt=max(total_attempts, 1))\n            return outcome\n        finally:\n            await events.put(_EVENTS_DONE)\n'


In [ ]:
SESSION_SOURCE = '"""Application-facing control for one in-memory agent conversation."""\n\nfrom __future__ import annotations\n\nimport asyncio\nfrom collections import deque\nfrom dataclasses import dataclass\nfrom collections.abc import AsyncIterator, Sequence\nfrom enum import Enum\nfrom typing import cast\n\nfrom .model import AgentMessage, Role, StopReason\nfrom .runtime import AgentRunHandle, AgentRuntime, AssistantOutcome, RuntimeEvent\n\n\n_SESSION_EVENTS_DONE = object()\n\n\nclass SessionBusyError(RuntimeError):\n    """Raised when a Session already owns an active Agent Run."""\n\n\nclass InputKind(str, Enum):\n    STEERING = "steering"\n    FOLLOW_UP = "follow_up"\n\n\n@dataclass(frozen=True, slots=True)\nclass PendingInput:\n    kind: InputKind\n    message: AgentMessage\n\n\n@dataclass(frozen=True, slots=True)\nclass SessionRunResult:\n    outcome: AssistantOutcome\n    outcomes: tuple[AssistantOutcome, ...] = ()\n    pending_inputs: tuple[PendingInput, ...] = ()\n\n\nclass SessionRunHandle:\n    def __init__(\n        self,\n        task: asyncio.Task[SessionRunResult],\n        session: "AgentSession",\n        events: asyncio.Queue[RuntimeEvent | object],\n    ) -> None:\n        self._task = task\n        self._session = session\n        self._events = events\n\n    async def result(self) -> SessionRunResult:\n        return await self._task\n\n    def cancel(self) -> None:\n        if not self._task.done():\n            self._session.cancel()\n\n    async def events(self) -> AsyncIterator[RuntimeEvent]:\n        while True:\n            event = await self._events.get()\n            if event is _SESSION_EVENTS_DONE:\n                break\n            yield cast(RuntimeEvent, event)\n\n\nclass _SteeringQueue:\n    def __init__(self) -> None:\n        self._messages: deque[PendingInput] = deque()\n\n    def append(self, message: AgentMessage) -> None:\n        self._messages.append(PendingInput(InputKind.STEERING, message))\n\n    def pending(self) -> bool:\n        return bool(self._messages)\n\n    def take(self) -> Sequence[AgentMessage]:\n        return (self._messages.popleft().message,)\n\n    def drain(self) -> tuple[PendingInput, ...]:\n        drained = tuple(self._messages)\n        self._messages.clear()\n        return drained\n\n\nclass AgentSession:\n    """Keep one active Run and its lifecycle behind a small interface."""\n\n    def __init__(self, runtime: AgentRuntime) -> None:\n        if not isinstance(runtime, AgentRuntime):\n            raise TypeError("runtime must be an AgentRuntime")\n        self._runtime = runtime\n        self._busy = False\n        self._active: AgentRunHandle | None = None\n        self._cancel_requested = False\n        self._steering = _SteeringQueue()\n        self._follow_ups: deque[PendingInput] = deque()\n\n    @property\n    def busy(self) -> bool:\n        return self._busy\n\n    def start(self, prompt: str | AgentMessage) -> SessionRunHandle:\n        if self._busy:\n            raise SessionBusyError("Session already has an active Run")\n        message = (\n            AgentMessage.text(Role.USER, prompt) if isinstance(prompt, str) else prompt\n        )\n        if not isinstance(message, AgentMessage) or message.role is not Role.USER:\n            raise TypeError("prompt must be text or a user AgentMessage")\n        self._busy = True\n        self._cancel_requested = False\n        events: asyncio.Queue[RuntimeEvent | object] = asyncio.Queue()\n        task = asyncio.get_running_loop().create_task(self._drive(message, events))\n        return SessionRunHandle(task, self, events)\n\n    async def run(self, prompt: str | AgentMessage) -> SessionRunResult:\n        return await self.start(prompt).result()\n\n    async def prompt(self, prompt: str | AgentMessage) -> SessionRunResult:\n        return await self.run(prompt)\n\n    def steer(self, message: str | AgentMessage) -> None:\n        if not self._busy:\n            raise RuntimeError("Steering requires an active Run")\n        self._steering.append(self._user_message(message))\n\n    def follow_up(self, message: str | AgentMessage) -> None:\n        if not self._busy:\n            raise RuntimeError("Follow-up requires an active Run")\n        self._follow_ups.append(\n            PendingInput(InputKind.FOLLOW_UP, self._user_message(message))\n        )\n\n    def cancel(self) -> None:\n        self._cancel_requested = True\n        if self._active is not None:\n            self._active.cancel()\n\n    async def _drive(\n        self,\n        message: AgentMessage,\n        events: asyncio.Queue[RuntimeEvent | object],\n    ) -> SessionRunResult:\n        outcomes: list[AssistantOutcome] = []\n\n        async def forward(handle: AgentRunHandle) -> None:\n            async for event in handle.events():\n                await events.put(event)\n\n        try:\n            next_message = message\n            while True:\n                self._active = self._runtime.start(\n                    [next_message], turn_input=self._steering\n                )\n                if self._cancel_requested:\n                    self._active.cancel()\n                forwarding = asyncio.create_task(forward(self._active))\n                outcome = await self._active.result()\n                await forwarding\n                outcomes.append(outcome)\n                if outcome.stop_reason in {StopReason.ABORTED, StopReason.ERROR}:\n                    break\n                if not self._follow_ups:\n                    break\n                next_message = self._follow_ups.popleft().message\n            pending = self._steering.drain() + tuple(self._follow_ups)\n            self._follow_ups.clear()\n            return SessionRunResult(outcome, tuple(outcomes), pending)\n        finally:\n            self._active = None\n            self._cancel_requested = False\n            self._busy = False\n            await events.put(_SESSION_EVENTS_DONE)\n\n    @staticmethod\n    def _user_message(message: str | AgentMessage) -> AgentMessage:\n        accepted = (\n            AgentMessage.text(Role.USER, message)\n            if isinstance(message, str)\n            else message\n        )\n        if not isinstance(accepted, AgentMessage) or accepted.role is not Role.USER:\n            raise TypeError("Session input must be text or a user AgentMessage")\n        return accepted\n'


In [ ]:
INIT_SOURCE = 'from .model import (\n    AgentMessage,\n    ContentBlock,\n    ModelAdapter,\n    ModelAdapterError,\n    ModelEnd,\n    ModelError,\n    ModelErrorCode,\n    ModelEvent,\n    ModelMessage,\n    ModelProtocolError,\n    ModelRequest,\n    ModelResult,\n    ModelSpec,\n    ModelToolResultMessage,\n    OpenAICompatibleAdapter,\n    OpenAICompatibleConfig,\n    Role,\n    ScriptedModelAdapter,\n    StopReason,\n    TextContent,\n    TextDelta,\n    ToolCallContent,\n    ToolCallDelta,\n    UnsupportedContentError,\n    Usage,\n    UsageUpdate,\n    complete,\n    to_model_messages,\n)\nfrom .runtime import (\n    AgentRunHandle,\n    AgentRuntime,\n    AssistantOutcome,\n    EventType,\n    RetryPolicy,\n    RunGuard,\n    RuntimeEvent,\n    Sleeper,\n    TerminalStatus,\n    TurnInput,\n)\nfrom .session import (\n    AgentSession,\n    InputKind,\n    PendingInput,\n    SessionBusyError,\n    SessionRunHandle,\n    SessionRunResult,\n)\nfrom .tools import (\n    AsyncioProcessOperations,\n    CompleteOutputKind,\n    CompleteOutputReference,\n    LocalToolExecutor,\n    PreparedToolCall,\n    ProcessResult,\n    Tool,\n    ToolErrorCode,\n    ToolExecutor,\n    ToolOutputBudget,\n    ToolResult,\n    ToolResultMessage,\n    TruncationDirection,\n    TruncationNotice,\n)\n\n__all__ = [name for name in globals() if not name.startswith("_")]\n'


In [ ]:
import asyncio
import importlib
from tempfile import TemporaryDirectory

MODEL_SOURCE = (CHAPTER_3 / 'src' / 'agent_harness' / 'model.py').read_text(encoding='utf-8')
with TemporaryDirectory(prefix='chapter-04-minimal-') as temporary:
    package = Path(temporary) / 'agent_harness'
    package.mkdir()
    (package / 'model.py').write_text(MODEL_SOURCE, encoding='utf-8')
    (package / 'tools.py').write_text(TOOLS_SOURCE, encoding='utf-8')
    (package / 'runtime.py').write_text(RUNTIME_SOURCE, encoding='utf-8')
    (package / 'session.py').write_text(SESSION_SOURCE, encoding='utf-8')
    (package / '__init__.py').write_text(INIT_SOURCE, encoding='utf-8')
    sys.path.insert(0, temporary)
    try:
        chapter4 = importlib.import_module('agent_harness')

        async def minimal_session_run():
            adapter = chapter4.ScriptedModelAdapter([
                [chapter4.TextDelta('initial path'), chapter4.ModelEnd(chapter4.StopReason.COMPLETE)],
                [chapter4.TextDelta('redirected path'), chapter4.ModelEnd(chapter4.StopReason.COMPLETE)],
            ])
            session = chapter4.AgentSession(chapter4.AgentRuntime(
                adapter, chapter4.ModelSpec('scripted/chapter-04')
            ))
            handle = session.start('begin')
            session.steer('take the safer path')
            result = await handle.result()
            events = [event async for event in handle.events()]
            return result, events, adapter.received_requests

        minimal_result, minimal_events, minimal_requests = await minimal_session_run()
    finally:
        sys.path.pop(0)
        for module_name in tuple(sys.modules):
            if module_name == 'agent_harness' or module_name.startswith('agent_harness.'):
                del sys.modules[module_name]

assert minimal_result.outcome.message.content[0].text == 'redirected path'
assert len(minimal_requests) == 2
assert minimal_requests[1].messages[-1].content[0].text == 'take the safer path'


## Staged Construction

The first tracer bullet drove one accepted prompt through `AgentSession` and proved that a competing prompt fails explicitly. Successive red–green slices added turn-boundary Steering, post-`agent_end` Follow-up execution, independent Sessions, cancellation settlement for parallel Tools and real processes, typed pending inputs, ordered Session Events, and four opt-in RunGuard outcomes. Each slice used only public interfaces and kept every prior Checkpoint test green.

The queues deliberately have different ownership. Runtime can take one Steering Message only while advancing the active Run; Session takes a Follow-up only after that Runtime handle has fully settled. This keeps message delivery deterministic without serializing unrelated Sessions.

In [ ]:
RUN_CONTROL_TEST_SOURCE = 'from __future__ import annotations\n\nimport asyncio\nimport sys\n\nimport pytest\n\nfrom agent_harness import (\n    AgentMessage,\n    AgentRuntime,\n    AgentSession,\n    AsyncioProcessOperations,\n    EventType,\n    InputKind,\n    ModelEnd,\n    ModelRequest,\n    ModelSpec,\n    Role,\n    RunGuard,\n    SessionBusyError,\n    StopReason,\n    TerminalStatus,\n    TextDelta,\n    Tool,\n    ToolCallDelta,\n    ToolErrorCode,\n    ToolResult,\n    ToolResultMessage,\n    Usage,\n    UsageUpdate,\n)\n\n\ndef test_session_rejects_a_second_prompt_while_one_run_is_active() -> None:\n    class BlockingAdapter:\n        def __init__(self) -> None:\n            self.started = asyncio.Event()\n            self.release = asyncio.Event()\n\n        async def stream(self, request: ModelRequest):\n            self.started.set()\n            await self.release.wait()\n            yield TextDelta("first complete")\n            yield ModelEnd(StopReason.COMPLETE)\n\n    async def scenario() -> None:\n        adapter = BlockingAdapter()\n        session = AgentSession(\n            AgentRuntime(adapter, ModelSpec("scripted/one-active-run"))\n        )\n        first = session.start("first")\n        await adapter.started.wait()\n\n        assert session.busy is True\n        with pytest.raises(SessionBusyError, match="active Run"):\n            await session.prompt("second")\n\n        adapter.release.set()\n        result = await first.result()\n\n        assert result.outcome.message == AgentMessage.text(\n            Role.ASSISTANT, "first complete"\n        )\n        assert session.busy is False\n\n    asyncio.run(scenario())\n\n\ndef test_steering_is_consumed_at_the_next_turn_boundary() -> None:\n    class SteerableAdapter:\n        def __init__(self) -> None:\n            self.requests: list[ModelRequest] = []\n            self.first_started = asyncio.Event()\n            self.release_first = asyncio.Event()\n\n        async def stream(self, request: ModelRequest):\n            self.requests.append(request)\n            if len(self.requests) == 1:\n                self.first_started.set()\n                await self.release_first.wait()\n                yield TextDelta("initial direction")\n            else:\n                yield TextDelta("redirected result")\n            yield ModelEnd(StopReason.COMPLETE)\n\n    async def scenario() -> None:\n        adapter = SteerableAdapter()\n        session = AgentSession(\n            AgentRuntime(adapter, ModelSpec("scripted/steering"))\n        )\n        handle = session.start("begin")\n        await adapter.first_started.wait()\n\n        session.steer("use the safer route")\n        adapter.release_first.set()\n        result = await handle.result()\n\n        assert result.outcome.message == AgentMessage.text(\n            Role.ASSISTANT, "redirected result"\n        )\n        assert len(adapter.requests) == 2\n        assert adapter.requests[1].messages[-1] == AgentMessage.text(\n            Role.USER, "use the safer route"\n        ).to_model()\n        assert result.pending_inputs == ()\n\n    asyncio.run(scenario())\n\n\ndef test_follow_up_starts_a_new_run_only_after_the_current_run_settles() -> None:\n    class FollowUpAdapter:\n        def __init__(self) -> None:\n            self.requests: list[ModelRequest] = []\n            self.first_started = asyncio.Event()\n            self.release_first = asyncio.Event()\n\n        async def stream(self, request: ModelRequest):\n            self.requests.append(request)\n            if len(self.requests) == 1:\n                self.first_started.set()\n                await self.release_first.wait()\n                yield TextDelta("first settled")\n            else:\n                yield TextDelta("follow-up settled")\n            yield ModelEnd(StopReason.COMPLETE)\n\n    async def scenario() -> None:\n        adapter = FollowUpAdapter()\n        session = AgentSession(\n            AgentRuntime(adapter, ModelSpec("scripted/follow-up"))\n        )\n        handle = session.start("first")\n        await adapter.first_started.wait()\n\n        session.follow_up("then inspect the result")\n        await asyncio.sleep(0)\n        assert len(adapter.requests) == 1\n\n        adapter.release_first.set()\n        result = await handle.result()\n\n        assert [outcome.message for outcome in result.outcomes] == [\n            AgentMessage.text(Role.ASSISTANT, "first settled"),\n            AgentMessage.text(Role.ASSISTANT, "follow-up settled"),\n        ]\n        assert adapter.requests[1].messages[-2:] == (\n            AgentMessage.text(Role.ASSISTANT, "first settled").to_model(),\n            AgentMessage.text(Role.USER, "then inspect the result").to_model(),\n        )\n        assert result.pending_inputs == ()\n\n    asyncio.run(scenario())\n\n\ndef test_different_sessions_run_independently() -> None:\n    class GateAdapter:\n        def __init__(self, answer: str) -> None:\n            self.answer = answer\n            self.started = asyncio.Event()\n            self.release = asyncio.Event()\n\n        async def stream(self, request: ModelRequest):\n            self.started.set()\n            await self.release.wait()\n            yield TextDelta(self.answer)\n            yield ModelEnd(StopReason.COMPLETE)\n\n    async def scenario() -> None:\n        left_adapter = GateAdapter("left")\n        right_adapter = GateAdapter("right")\n        left = AgentSession(\n            AgentRuntime(left_adapter, ModelSpec("scripted/left"))\n        )\n        right = AgentSession(\n            AgentRuntime(right_adapter, ModelSpec("scripted/right"))\n        )\n\n        left_handle = left.start("one")\n        right_handle = right.start("two")\n        await asyncio.gather(left_adapter.started.wait(), right_adapter.started.wait())\n\n        right_adapter.release.set()\n        right_result = await asyncio.wait_for(right_handle.result(), timeout=1)\n        assert right_result.outcome.message == AgentMessage.text(Role.ASSISTANT, "right")\n        assert left.busy is True\n\n        left_adapter.release.set()\n        left_result = await asyncio.wait_for(left_handle.result(), timeout=1)\n        assert left_result.outcome.message == AgentMessage.text(Role.ASSISTANT, "left")\n\n    asyncio.run(scenario())\n\n\ndef test_cancellation_settles_parallel_tools_and_returns_unconsumed_input() -> None:\n    async def scenario() -> None:\n        quick_finished = asyncio.Event()\n        slow_started = asyncio.Event()\n\n        async def quick(arguments: dict[str, object]) -> ToolResult:\n            quick_finished.set()\n            return ToolResult("quick result")\n\n        async def slow(arguments: dict[str, object]) -> ToolResult:\n            slow_started.set()\n            await asyncio.Event().wait()\n            return ToolResult("unreachable")\n\n        adapter = type(\n            "ToolAdapter",\n            (),\n            {\n                "stream": lambda self, request: _tool_call_stream(),\n            },\n        )()\n\n        runtime = AgentRuntime(\n            adapter,\n            ModelSpec("scripted/cancel-tools"),\n            tools=[\n                Tool("quick", "Finish quickly", {"type": "object"}, quick),\n                Tool("slow", "Wait for cancellation", {"type": "object"}, slow),\n            ],\n        )\n        session = AgentSession(runtime)\n        handle = session.start("run both")\n        await asyncio.gather(quick_finished.wait(), slow_started.wait())\n        await asyncio.sleep(0)\n\n        session.steer("queued steering")\n        session.follow_up("queued follow-up")\n        events_task = asyncio.create_task(\n            _collect_cancellation_events(handle)\n        )\n        handle.cancel()\n        result = await asyncio.wait_for(handle.result(), timeout=1)\n        events = await events_task\n\n        assert result.outcome.stop_reason is StopReason.ABORTED\n        assert result.outcome.status is TerminalStatus.CANCELLED\n        tool_messages = [\n            item for item in runtime.history if isinstance(item, ToolResultMessage)\n        ]\n        assert [item.tool_name for item in tool_messages] == ["quick", "slow"]\n        assert tool_messages[0].result == ToolResult("quick result")\n        assert tool_messages[1].result.error_code is ToolErrorCode.CANCELLED\n        slow_end = next(\n            event\n            for event in events\n            if event.type is EventType.TOOL_CALL_END\n            and event.tool_call_id == "slow-id"\n        )\n        assert slow_end.tool_result is not None\n        assert slow_end.tool_result.error_code is ToolErrorCode.CANCELLED\n        assert runtime.history[-1] == result.outcome.message\n        assert [item.kind for item in result.pending_inputs] == [\n            InputKind.STEERING,\n            InputKind.FOLLOW_UP,\n        ]\n        assert [item.message for item in result.pending_inputs] == [\n            AgentMessage.text(Role.USER, "queued steering"),\n            AgentMessage.text(Role.USER, "queued follow-up"),\n        ]\n\n    async def _tool_call_stream():\n        yield ToolCallDelta(0, "quick-id", "quick", "{}")\n        yield ToolCallDelta(1, "slow-id", "slow", "{}")\n        yield ModelEnd(StopReason.TOOL_USE)\n\n    async def _collect_cancellation_events(handle):\n        return [event async for event in handle.events()]\n\n    asyncio.run(scenario())\n\n\ndef test_cancellation_terminates_a_real_process_operation(tmp_path) -> None:\n    async def scenario() -> None:\n        started = tmp_path / "started"\n        incorrectly_finished = tmp_path / "finished"\n        script = (\n            "from pathlib import Path; import time; "\n            "Path(\'started\').write_text(\'yes\'); "\n            "time.sleep(0.3); Path(\'finished\').write_text(\'should not happen\')"\n        )\n        operation = AsyncioProcessOperations()\n        task = asyncio.create_task(\n            operation.run([sys.executable, "-c", script], cwd=tmp_path)\n        )\n        for _ in range(100):\n            if started.exists():\n                break\n            await asyncio.sleep(0.01)\n        assert started.exists()\n\n        task.cancel()\n        with pytest.raises(asyncio.CancelledError):\n            await task\n        await asyncio.sleep(0.35)\n\n        assert not incorrectly_finished.exists()\n\n    asyncio.run(scenario())\n\n\n@pytest.mark.parametrize(\n    ("guard", "expected"),\n    [\n        (RunGuard(max_turns=1), TerminalStatus.MAX_TURNS),\n        (RunGuard(max_tool_calls=1), TerminalStatus.MAX_TOOL_CALLS),\n        (RunGuard(max_total_tokens=5), TerminalStatus.MAX_TOTAL_TOKENS),\n    ],\n)\ndef test_run_guards_stop_with_distinct_status_at_a_settled_boundary(\n    guard: RunGuard,\n    expected: TerminalStatus,\n) -> None:\n    class RepeatingToolAdapter:\n        def __init__(self) -> None:\n            self.turn = 0\n\n        async def stream(self, request: ModelRequest):\n            self.turn += 1\n            yield ToolCallDelta(self.turn - 1, f"call-{self.turn}", "step", "{}")\n            yield UsageUpdate(Usage(4, 1, 5))\n            yield ModelEnd(StopReason.TOOL_USE)\n\n    async def scenario() -> None:\n        async def step(arguments: dict[str, object]) -> ToolResult:\n            return ToolResult("settled")\n\n        adapter = RepeatingToolAdapter()\n        runtime = AgentRuntime(\n            adapter,\n            ModelSpec("scripted/guard"),\n            tools=[Tool("step", "Take one step", {"type": "object"}, step)],\n            run_guard=guard,\n        )\n\n        outcome = await runtime.run([AgentMessage.text(Role.USER, "continue")])\n\n        assert outcome.status is expected\n        assert outcome.stop_reason is StopReason.ABORTED\n        assert adapter.turn == 1\n        assert isinstance(runtime.history[-1], ToolResultMessage)\n        assert runtime.history[-1].tool_call_id == "call-1"\n\n    asyncio.run(scenario())\n\n\ndef test_timeout_guard_cancels_provider_work_and_settles_distinctly() -> None:\n    class BlockingAdapter:\n        async def stream(self, request: ModelRequest):\n            yield TextDelta("partial")\n            await asyncio.Event().wait()\n            yield ModelEnd(StopReason.COMPLETE)\n\n    async def scenario() -> None:\n        runtime = AgentRuntime(\n            BlockingAdapter(),\n            ModelSpec("scripted/timeout"),\n            run_guard=RunGuard(timeout_seconds=0.02),\n        )\n\n        outcome = await asyncio.wait_for(\n            runtime.run([AgentMessage.text(Role.USER, "wait")]), timeout=0.5\n        )\n\n        assert outcome.status is TerminalStatus.TIMEOUT\n        assert outcome.stop_reason is StopReason.ABORTED\n        assert outcome.message == AgentMessage.text(Role.ASSISTANT, "partial")\n        assert runtime.history[-1] == outcome.message\n\n    asyncio.run(scenario())\n\n\ndef test_session_handle_observes_follow_up_runs_in_lifecycle_order() -> None:\n    class TwoRunAdapter:\n        def __init__(self) -> None:\n            self.turn = 0\n            self.started = asyncio.Event()\n            self.release = asyncio.Event()\n\n        async def stream(self, request: ModelRequest):\n            self.turn += 1\n            if self.turn == 1:\n                self.started.set()\n                await self.release.wait()\n            yield TextDelta(f"answer {self.turn}")\n            yield ModelEnd(StopReason.COMPLETE)\n\n    async def scenario() -> None:\n        adapter = TwoRunAdapter()\n        session = AgentSession(\n            AgentRuntime(adapter, ModelSpec("scripted/session-events"))\n        )\n        handle = session.start("first")\n        await adapter.started.wait()\n        session.follow_up("second")\n        events_task = asyncio.create_task(_collect(handle))\n        adapter.release.set()\n\n        await handle.result()\n        event_types = [event.type for event in await events_task]\n\n        first_end = event_types.index(EventType.AGENT_END)\n        second_start = event_types.index(EventType.AGENT_START, first_end + 1)\n        assert first_end < second_start\n        assert event_types.count(EventType.AGENT_START) == 2\n        assert event_types.count(EventType.AGENT_END) == 2\n\n    async def _collect(handle):\n        return [event async for event in handle.events()]\n\n    asyncio.run(scenario())\n\n\ndef test_session_immediate_cancel_still_settles_the_accepted_run() -> None:\n    class BlockingAdapter:\n        async def stream(self, request: ModelRequest):\n            await asyncio.Event().wait()\n            yield ModelEnd(StopReason.COMPLETE)\n\n    async def scenario() -> None:\n        session = AgentSession(\n            AgentRuntime(BlockingAdapter(), ModelSpec("scripted/session-cancel"))\n        )\n        handle = session.start("cancel now")\n\n        handle.cancel()\n        result = await asyncio.wait_for(handle.result(), timeout=0.2)\n\n        assert result.outcome.status is TerminalStatus.CANCELLED\n        assert result.outcome.stop_reason is StopReason.ABORTED\n        assert session.busy is False\n\n    asyncio.run(scenario())\n\n\ndef test_completed_handle_cannot_cancel_a_later_run() -> None:\n    class TwoRunAdapter:\n        def __init__(self) -> None:\n            self.turn = 0\n            self.second_started = asyncio.Event()\n            self.release_second = asyncio.Event()\n\n        async def stream(self, request: ModelRequest):\n            self.turn += 1\n            if self.turn == 2:\n                self.second_started.set()\n                await self.release_second.wait()\n            yield TextDelta(f"answer {self.turn}")\n            yield ModelEnd(StopReason.COMPLETE)\n\n    async def scenario() -> None:\n        adapter = TwoRunAdapter()\n        session = AgentSession(\n            AgentRuntime(adapter, ModelSpec("scripted/stale-handle"))\n        )\n        completed_handle = session.start("first")\n        await completed_handle.result()\n\n        later_handle = session.start("second")\n        await adapter.second_started.wait()\n        completed_handle.cancel()\n        await asyncio.sleep(0)\n\n        assert session.busy is True\n        adapter.release_second.set()\n        later = await later_handle.result()\n        assert later.outcome.status is TerminalStatus.COMPLETED\n        assert later.outcome.message == AgentMessage.text(Role.ASSISTANT, "answer 2")\n\n    asyncio.run(scenario())\n'


In [ ]:
CHAPTER_CONTRACT_TEST_SOURCE = 'from __future__ import annotations\n\nimport asyncio\n\nimport pytest\n\nfrom agent_harness import AsyncioProcessOperations, RunGuard\n\n\ndef test_run_guard_and_process_inputs_fail_before_work_is_accepted() -> None:\n    for name in ("max_turns", "max_tool_calls", "max_total_tokens"):\n        with pytest.raises(ValueError, match=name):\n            RunGuard(**{name: 0})\n    with pytest.raises(ValueError, match="timeout_seconds"):\n        RunGuard(timeout_seconds=0)\n\n    async def scenario() -> None:\n        operation = AsyncioProcessOperations()\n        with pytest.raises(ValueError, match="command"):\n            await operation.run([])\n        with pytest.raises(ValueError, match="timeout_seconds"):\n            await operation.run(["not-started"], timeout_seconds=0)\n\n    asyncio.run(scenario())\n'


## Observable Trace

`SessionRunHandle.events()` forwards each active Runtime's ordered observational stream. A Follow-up therefore exposes a complete `agent_end` before the next `agent_start`; Steering stays inside one Run and adds another model turn without fabricating a lifecycle transition.

In [ ]:
observable_trace = [
    {'sequence': event.sequence, 'type': event.type.value}
    for event in minimal_events
]
event_types = [item['type'] for item in observable_trace]
assert event_types.count('agent_start') == 1
assert event_types.count('agent_end') == 1
assert event_types.count('model_attempt_start') == 2
observable_trace


## Failure Boundaries and Trade-offs

A first cancellation request is coordinated rather than destructive. Partial provider text becomes an aborted Assistant outcome. During a Tool Batch, completed results survive and each unfinished call receives a concise `cancelled` ToolResult before the final aborted outcome, so history never contains an orphan Tool Call. Steering and Follow-up inputs that never entered history return as typed `pending_inputs`.

Run guards are absent by default. Maximum turns, Tool calls, and total tokens are checked only where continuing would cross a settled boundary; an indivisible Tool Batch completes before a guard stops continuation. A timeout requests the same coordinated cancellation path but records the distinct `timeout` status. `AsyncioProcessOperations` terminates a real host child process and explicitly provides no sandbox guarantee.

In [ ]:
async def cancellation_demo():
    class BlockingAdapter:
        def __init__(self):
            self.started = asyncio.Event()

        async def stream(self, request):
            yield chapter4.TextDelta('partial')
            self.started.set()
            await asyncio.Event().wait()
            yield chapter4.ModelEnd(chapter4.StopReason.COMPLETE)

    adapter = BlockingAdapter()
    session = chapter4.AgentSession(chapter4.AgentRuntime(
        adapter, chapter4.ModelSpec('scripted/cancellation-demo')
    ))
    handle = session.start('start work')
    await adapter.started.wait()
    session.steer('not consumed')
    session.follow_up('also not consumed')
    handle.cancel()
    return await handle.result()

cancelled_result = await cancellation_demo()
assert cancelled_result.outcome.status is chapter4.TerminalStatus.CANCELLED
assert cancelled_result.outcome.stop_reason is chapter4.StopReason.ABORTED
assert [item.kind for item in cancelled_result.pending_inputs] == [
    chapter4.InputKind.STEERING, chapter4.InputKind.FOLLOW_UP
]


## Checkpoint Export and Verification

Chapter metadata names Chapter 3 as the base Checkpoint. Export carries forward all cumulative sources and regression tests, replaces the evolved modules, adds Session control tests, writes a deterministic manifest, then compiles, installs without dependency resolution, imports, and runs every test before publishing Chapter 4.

In [ ]:
PYPROJECT_SOURCE = '[build-system]\nrequires = ["setuptools>=68"]\nbuild-backend = "setuptools.build_meta"\n\n[project]\nname = "agent-harness"\nversion = "0.4.0"\ndescription = "Chapter 4 controlled Agent Sessions and cancellation settlement"\nreadme = "README.md"\nrequires-python = ">=3.11"\ndependencies = ["jsonschema>=4.23,<5", "openai>=1.40,<3"]\n\n[tool.setuptools.packages.find]\nwhere = ["src"]\n\n[tool.setuptools.package-data]\nagent_harness = ["py.typed"]\n\n[tool.pytest.ini_options]\ntestpaths = ["tests"]\n'


In [ ]:
README_SOURCE = '# Agent Harness — Chapter 4 Checkpoint\n\nThis cumulative Checkpoint adds `AgentSession`, the application-facing control module around `AgentRuntime`. A Session accepts one active Run, rejects a competing prompt explicitly, injects Steering Messages at the next valid turn boundary, and starts queued Follow-up Messages only after the preceding Run has emitted `agent_end`. Independent Sessions remain concurrently runnable.\n\n`SessionRunHandle` exposes ordered Events, coordinated cancellation, and an eventual `SessionRunResult`. Cancellation propagates through provider streaming and Tool execution. Completed parallel Tool results are retained, unfinished calls receive structured `cancelled` ToolResults in source order, an aborted Assistant outcome closes the history, and unconsumed Steering and Follow-up inputs return as typed `pending_inputs`.\n\n`RunGuard` provides opt-in maximum turns, Tool calls, elapsed time, and total-token limits with distinct terminal statuses. There are no aggregate limits by default. Guard and cancellation exits occur only after coherent outcomes are materialized. `AsyncioProcessOperations` demonstrates cancellation of a real host child process without claiming sandbox isolation.\n\nAll ordinary tests are deterministic and offline. The inherited real-endpoint smoke remains supplementary and explicitly credential-gated.\n'


In [ ]:
from course.tools.checkpoint import checkpoint_drift, export_checkpoint

checkpoint_result = export_checkpoint(
    ROOT / 'course' / 'notebooks' / '04_run_control.ipynb',
    ROOT / 'course' / 'checkpoints' / 'ch04',
)
assert checkpoint_result.gates == ('compile', 'install', 'import', 'tests')
assert checkpoint_drift(
    ROOT / 'course' / 'notebooks' / '04_run_control.ipynb',
    ROOT / 'course' / 'checkpoints' / 'ch04',
) == ()
checkpoint_result


## Public API Summary

Construct `AgentSession(AgentRuntime(...))`, then use `session.start(prompt)` for Events and cancellation or `await session.run(prompt)` / `await session.prompt(prompt)` for completion. While `session.busy`, call `session.steer(message)` for the next turn boundary or `session.follow_up(message)` for a later Run; a competing prompt raises `SessionBusyError`.

Inspect `SessionRunResult.outcome`, cumulative `outcomes`, and typed `pending_inputs`. Configure no aggregate limit by default or pass `RunGuard(max_turns=..., max_tool_calls=..., timeout_seconds=..., max_total_tokens=...)`. `TerminalStatus`, `InputKind`, `PendingInput`, `SessionRunHandle`, `TurnInput`, `AsyncioProcessOperations`, and `ProcessResult` are public contracts.